# Context Control & Scoping (`window.craftContext`)

> Transactional visibility engine for Solveit's per-cell hide-from-AI flags: **All-But (1.1)**, **Context Lens (1.2)**, **Saved Context Presets (1.3)**, **Context Budget Meter (1.4)**, **"What the AI Sees" Preview (1.5)**, and **Traceback Quick-Fix (1.6)**.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'ctx_staging_bar': Div(
        Span('', cls='craft-ab-summary', data_ref='summary'),
        Select(Option('Presets…', value=''), cls='uk-select craft-ab-preset', aria_label='Context presets', data_ref='preset'),
        craft_btn('Save preset', size='xs', cls='craft-ab-save', data_ref='save'),
        craft_btn('Apply perm', size='xs', cls='craft-ab-perm', data_ref='perm'),
        craft_btn('Run prompt', variant='primary', size='xs', cls='craft-ab-run', data_ref='run'),
        craft_btn('Cancel', size='xs', cls='craft-ab-cancel', data_ref='cancel'),
        id='craft-ab-bar', role='region', aria_label='All-But context staging',
    ),
    'ctx_recover_bar': Div(
        Span('Pre-All-But cell visibility snapshot detected.', cls='craft-ab-rec-msg', data_ref='msg'),
        craft_btn('Restore pre-All-But visibility', variant='primary', size='xs', data_ref='restore'),
        craft_btn('Dismiss', size='xs', data_ref='dismiss'),
        id='craft-ab-recover', role='alert',
    ),
    'ctx_meter': Button(
        Span('0 tok', cls='craft-ctx-meter-val', data_ref='val'),
        id='craft-ctx-meter', type='button', cls='craft-ctx-meter',
        title='Active AI context budget (click to preview what the AI sees)',
        aria_label='Active AI context budget',
    ),
    'ctx_preview_modal': craft_overlay(
        craft_card(
            Div(
                Strong('What the AI Sees'),
                Span('', cls='craft-ctx-prev-total', data_ref='total'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(
                'Approximation of dialog cells sent to the AI. Server-side system prompts, tool schemas, and CRAFT.ipynb injections are not inspectable from the client.',
                cls='craft-ctx-prev-note',
            ),
            Div(cls='craft-ctx-prev-list', data_ref='list'),
            Div(
                craft_btn('Toggle heat strip', size='xs', data_ref='heat'),
                craft_btn('Save as preset', size='xs', data_ref='savePreset'),
                craft_btn('Close', variant='primary', size='xs', data_ref='done'),
                cls='craft-modal-foot',
            ),
            cls='craft-ctx-prev-card',
        ),
        id='craft-ai-preview', aria_label='What the AI Sees preview',
    ),
    'ctx_tb_fix_btn': Button(
        'Ask AI about this error',
        type='button', cls='craft-tb-fix-btn',
        title='Insert a prompt below and run All-But scoped to this failing cell and traceback',
    ),
}

## Cell Inspection, Token Estimation & Presets (`#| export_js_iife`)


In [ ]:
#| export_js_iife CRAFT.js craftContext
#| eval: false
window.craftContext?.disable?.();
if (!document.getElementById('dialog-container')) return;

const CORE = window.__craftCMCore;
if (!CORE || !CORE.features.init('context-control', 'Context Control', 'All-But scoping, Context Lenses, Presets, Budget Meter, Traceback Quick-Fix')) return;
const {
  el, qs, qsa, on, store, mountTpl, mountModal, htmxPost, onCells, onDomChange, SOLVEIT,
  notify, notifyErr, kid, normId, getCellEl, typingBlocked,
  allCells, selectedCells, primaryCell, cellMtype, dlgKey,
  waitForHtmxSettle, waitForNewCell, onCellHtmx, headingLevel, mountBar,
} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};

const snapKey = () => `craft-allbut:${kid() || dlgKey()}`;
const presetKey = () => `craft-ctx-presets:${dlgKey()}`;
const HEAT_KEY = 'craft-ctx-heat';

const isHiddenFromAI = cell => !!cell?.querySelector(SOLVEIT.sel.skipActive);
const isPinned = cell => !!cell?.querySelector('button[hx-post="/toggle_pin_"] use[href="#lc-pin"]');
const hasTraceback = cell => {
  const out = cell?.querySelector(SOLVEIT.sel.cardOut);
  if (!out) return false;
  if (out.matches(`:has(${SOLVEIT.sel.ansiRed}, .ansi-red-intense-fg)`)) return true;
  return /Traceback \(most recent call last\)|Error:|Exception:/i.test(out.textContent || '');
};

const parseTokenNum = raw => {
  const s = String(raw || '').replace(/,/g, '').trim();
  const m = s.match(/(\d+(?:\.\d+)?)\s*([kKmM])?/);
  if (!m) return 0;
  let n = parseFloat(m[1]);
  if (m[2]?.toLowerCase() === 'k') n *= 1000;
  else if (m[2]?.toLowerCase() === 'm') n *= 1e6;
  return Math.round(n);
};

const cellTokens = cell => {
  if (!cell) return 0;
  const attrs = qsa('[data-tokens]', cell);
  if (attrs.length) {
    let sum = 0;
    for (const a of attrs) sum += parseTokenNum(a.getAttribute('data-tokens'));
    if (sum > 0) return sum;
  }
  const hdrs = qsa(`:scope > .msg-card > ${SOLVEIT.sel.cardHeader} nav > div.grow`, cell);
  let sum = 0;
  for (const h of hdrs) {
    const m = (h.textContent || '').match(/(?:Code|Note|Prompt|Raw|Output|Assistant|You)\s*[:·]\s*([\d,]+)/i);
    if (m) sum += parseTokenNum(m[1]);
  }
  if (sum > 0) return sum;
  const txt = (cell.textContent || '').trim();
  return txt ? Math.max(1, Math.ceil(txt.length / 4)) : 0;
};

const fmtTok = n => n >= 1000 ? `${(n / 1000).toFixed(1)}k` : String(n);

async function setHiddenFromAI(cell, wantHidden) {
  if (!cell || !cell.isConnected) return false;
  if (isHiddenFromAI(cell) === wantHidden) return true;
  const btn = cell.querySelector(SOLVEIT.sel.skipBtn);
  if (btn) {
    btn.click();
  } else {
    try { await htmxPost(SOLVEIT.api.toggleSkip, {id_: cell.id, ids: cell.id}); }
    catch (err) { console.error('craftContext: toggle_skip failed', err); return false; }
  }
  return Boolean(await waitForHtmxSettle(() => {
    const cur = document.getElementById(cell.id);
    return cur && isHiddenFromAI(cur) === wantHidden;
  }, 1200));
}

const getHeadingLevel = cell => {
  const sh = cell?.getAttribute('data-sm-header');
  return (sh && /^[1-6]$/.test(sh)) ? +sh : headingLevel(cell);
};

const getSectionCells = startCell => {
  const all = allCells();
  const idx = all.indexOf(startCell);
  if (idx < 0) return [];
  let headIdx = idx, headLvl = getHeadingLevel(startCell);
  if (!headLvl) {
    for (let i = idx - 1; i >= 0; i--) {
      const l = getHeadingLevel(all[i]);
      if (l) { headIdx = i; headLvl = l; break; }
    }
  }
  if (!headLvl) return [startCell];
  const out = [all[headIdx]];
  for (let i = headIdx + 1; i < all.length; i++) {
    const l = getHeadingLevel(all[i]);
    if (l && l <= headLvl) break;
    out.push(all[i]);
  }
  return out;
};

/* ── Saved Context Presets (1.3) ───────────────────────────────────── */
const loadPresets = () => {
  const raw = store.json(presetKey(), {}) || {};
  const existing = new Set(allCells().map(c => normId(c.id)));
  const cleaned = {};
  let mutated = false;
  for (const [name, ids] of Object.entries(raw)) {
    if (!Array.isArray(ids)) continue;
    const valid = ids.map(normId).filter(id => !existing.size || existing.has(id));
    if (valid.length !== ids.length) mutated = true;
    cleaned[name] = valid;
  }
  if (mutated && existing.size) store.setJson(presetKey(), cleaned);
  return cleaned;
};

const savePreset = (name, ids) => {
  const n = String(name || '').trim();
  if (!n) return false;
  const presets = loadPresets();
  presets[n] = [...new Set(ids.map(normId))];
  store.setJson(presetKey(), presets);
  return true;
};

const deletePreset = name => {
  const presets = loadPresets();
  if (!(name in presets)) return false;
  delete presets[name];
  store.setJson(presetKey(), presets);
  return true;
};


## All-But Staging, Transactional Visibility & Context Lenses (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftContext
#| eval: false
/* ── Staging & Transactional Apply/Restore (1.1) ───────────────────── */
let staging = null; // {scopeIds: Set<id>, preHiddenIds: Set<id>, label: string, applying: bool, running: bool}
const stagingBar = mountBar('ctx_staging_bar'), recoverBar = mountBar('ctx_recover_bar');
let stagingWired = false, recoverWired = false, restoreWatch = null;

const readSnapshot = () => store.json(snapKey(), null, sessionStorage);
const writeSnapshot = snap => snap ? store.setJson(snapKey(), snap, sessionStorage) : store.del(snapKey(), sessionStorage);

function clearStagingVisuals() {
  document.documentElement.classList.remove('craft-ab-staging');
  for (const c of allCells()) c.classList.remove('craft-ab-included');
}

function syncStagingUI() {
  if (!staging) {
    clearStagingVisuals();
    stagingBar.hide();
    return;
  }
  document.documentElement.classList.add('craft-ab-staging');
  const cells = allCells();
  let scopeTok = 0, totalTok = 0;
  for (const c of cells) {
    const id = normId(c.id);
    const inc = staging.scopeIds.has(id);
    c.classList.toggle('craft-ab-included', inc);
    const t = cellTokens(c);
    totalTok += t;
    if (inc) scopeTok += t;
  }
  const b = stagingBar.show();
  if (!stagingWired) {
    stagingWired = true;
    on(b.cancel, 'click', () => cancelStaging(), sig);
    on(b.run, 'click', () => applyAndRun(), sig);
    on(b.perm, 'click', () => applyPermanent(), sig);
    on(b.save, 'click', () => {
      const name = prompt('Save current context selection as preset:');
      if (name && savePreset(name, [...staging.scopeIds])) {
        refreshPresetDropdown();
        notify(`Saved preset "${name}".`, 'success');
      }
    }, sig);
    on(b.preset, 'change', () => {
      const name = b.preset.value;
      if (!name) return;
      const presets = loadPresets();
      if (presets[name]) {
        staging.scopeIds = new Set(presets[name]);
        staging.label = `Preset: ${name}`;
        syncStagingUI();
      }
      b.preset.value = '';
    }, sig);
  }
  b.summary.textContent = `${staging.label}: ${staging.scopeIds.size} ${staging.scopeIds.size === 1 ? 'cell' : 'cells'} · ~${fmtTok(scopeTok)} / ${fmtTok(totalTok)} tokens · Click cells to toggle · Esc to cancel`;
  refreshPresetDropdown();
}

function refreshPresetDropdown() {
  const b = stagingBar.current;
  if (!b?.preset) return;
  const presets = loadPresets();
  b.preset.replaceChildren(
    el('option', {value: ''}, 'Presets…'),
    ...Object.keys(presets).sort().map(n => el('option', {value: n}, `${n} (${presets[n].length})`))
  );
}

function enterStaging(ids, label = 'All-But') {
  const scopeIds = new Set(ids.map(normId).filter(Boolean));
  const preHiddenIds = new Set(allCells().filter(isHiddenFromAI).map(c => normId(c.id)));
  staging = {scopeIds, preHiddenIds, label, applying: false, running: false};
  writeSnapshot({
    kid: kid(),
    preHiddenIds: [...preHiddenIds],
    scopeIds: [...scopeIds],
    label,
    applied: false,
    ts: Date.now(),
  });
  syncStagingUI();
}

function cancelStaging() {
  if (!staging) return;
  staging = null;
  writeSnapshot(null);
  syncStagingUI();
}

async function applyPermanent(ids = null) {
  const scope = new Set((ids || (staging ? [...staging.scopeIds] : [])).map(normId));
  if (!scope.size) return;
  staging = null;
  writeSnapshot(null);
  syncStagingUI();
  for (const c of allCells()) {
    const wantHidden = !scope.has(normId(c.id));
    if (isHiddenFromAI(c) !== wantHidden) await setHiddenFromAI(c, wantHidden);
  }
  updateMeter();
}

async function restoreFromSnapshot(snap = null) {
  const s = snap || readSnapshot();
  if (!s) return;
  const preHidden = new Set((s.preHiddenIds || []).map(normId));
  for (const c of allCells()) {
    const wantHidden = preHidden.has(normId(c.id));
    if (isHiddenFromAI(c) !== wantHidden) await setHiddenFromAI(c, wantHidden);
  }
  writeSnapshot(null);
  recoverBar.hide();
  staging = null;
  syncStagingUI();
  updateMeter();
}

function watchRunCompletion(snap) {
  if (restoreWatch) { restoreWatch(); restoreWatch = null; }
  let sawBusy = false, idleTicks = 0, done = false;
  const cleanup = () => {
    if (done) return;
    done = true;
    clearInterval(timer);
    offStop?.();
    offReq?.();
    restoreWatch = null;
  };
  const finish = async () => {
    if (done) return;
    cleanup();
    await restoreFromSnapshot(snap);
  };
  const isRunning = () => !!qs(`${SOLVEIT.sel.running}, ${SOLVEIT.sel.dialog} .uk-spinner`);
  const offReq = on(document, 'htmx:beforeRequest', e => {
    const p = String(e.detail?.path || '');
    if (p.includes(SOLVEIT.api.addRunq) || p.includes('/run_')) sawBusy = true;
  }, sig);
  const timer = setInterval(() => {
    if (isRunning()) { sawBusy = true; idleTicks = 0; }
    else {
      idleTicks++;
      if (sawBusy && idleTicks >= 3) finish();
      else if (!sawBusy && idleTicks >= 25) { cleanup(); checkRecoveryBanner(); }
    }
  }, 400);
  const offStop = on(window, 'keydown', e => {
    if (e.shiftKey && e.key.toLowerCase() === 's' && !typingBlocked()) setTimeout(finish, 300);
  }, {capture: true, signal: ac.signal});
  restoreWatch = cleanup;
}

async function applyDiffForOneShot() {
  if (!staging || staging.applying) return null;
  staging.applying = true;
  const snap = {
    kid: kid(),
    preHiddenIds: [...staging.preHiddenIds],
    scopeIds: [...staging.scopeIds],
    label: staging.label,
    applied: true,
    ts: Date.now(),
  };
  writeSnapshot(snap);
  const scope = staging.scopeIds;
  for (const c of allCells()) {
    const wantHidden = !scope.has(normId(c.id));
    if (isHiddenFromAI(c) !== wantHidden) await setHiddenFromAI(c, wantHidden);
  }
  staging.running = true;
  clearStagingVisuals();
  stagingBar.hide();
  staging = null;
  updateMeter();
  return snap;
}

async function applyAndRun(promptCellId = null) {
  if (!staging) return;
  const targetId = normId(promptCellId) ||
    [...staging.scopeIds].reverse().find(id => cellMtype(getCellEl(id)) === 'prompt') ||
    [...staging.scopeIds].at(-1);
  const snap = await applyDiffForOneShot();
  if (!snap) return;
  if (targetId && getCellEl(targetId)) {
    await CORE.runCellCmd({id: targetId, cmd: 'run', tag: 'craftContext'});
  }
  watchRunCompletion(snap);
}

function checkRecoveryBanner() {
  const snap = readSnapshot();
  if (!snap || !snap.applied || staging) { recoverBar.hide(); return; }
  const b = recoverBar.show();
  if (!recoverWired) {
    recoverWired = true;
    on(b.restore, 'click', () => restoreFromSnapshot(), sig);
    on(b.dismiss, 'click', () => { writeSnapshot(null); recoverBar.hide(); }, sig);
  }
}

/* Intercept clicks in staging mode to toggle cell inclusion, and Esc to cancel */
on(document, 'click', e => {
  if (!staging) return;
  if (stagingBar?.root?.contains(e.target)) return;
  const cell = e.target.closest?.(SOLVEIT.sel.cellWithId);
  if (!cell) return;
  e.preventDefault();
  e.stopImmediatePropagation();
  const id = normId(cell.id);
  if (staging.scopeIds.has(id)) staging.scopeIds.delete(id);
  else staging.scopeIds.add(id);
  syncStagingUI();
}, cap);

on(window, 'keydown', e => {
  if (!staging) return;
  if (e.key === 'Escape' && !e.isComposing) {
    e.preventDefault();
    e.stopImmediatePropagation();
    cancelStaging();
  }
}, cap);

/* Intercept prompt run requests while staging so normal Mod+Enter / Send button triggers All-But */
onCellHtmx([SOLVEIT.api.addRunq, SOLVEIT.api.continuePrompt], ({id, event}) => {
  if (!staging || staging.applying) return;
  if (id && cellMtype(getCellEl(id)) === 'prompt') {
    event.preventDefault();
    staging.scopeIds.add(id);
    applyAndRun(id);
  }
}, sig);

/* ── Context Lenses (1.2) ──────────────────────────────────────────── */
const lensAllBut = (cell = primaryCell()) => {
  const sel = selectedCells();
  const targets = sel.length > 1 ? sel : (cell ? [cell] : []);
  if (!targets.length) return notifyErr('Select a cell first for All-But.', 'craftContext');
  enterStaging(targets.map(c => c.id), 'All-But');
};

const lensHeadingSection = (cell = primaryCell()) => {
  const target = cell || primaryCell();
  if (!target) return notifyErr('Select a cell first.', 'craftContext');
  const sec = getSectionCells(target);
  enterStaging(sec.map(c => c.id), 'Section Lens');
};

const lensPinnedPlus = (cell = primaryCell()) => {
  const ids = allCells().filter(isPinned).map(c => c.id);
  if (cell) ids.push(cell.id);
  if (!ids.length) return notifyErr('No pinned or selected cells found.', 'craftContext');
  enterStaging(ids, 'Pinned + Cell');
};

const lensCodeAndErrors = () => {
  const ids = allCells().filter(c => cellMtype(c) === 'code' || hasTraceback(c)).map(c => c.id);
  if (!ids.length) return notifyErr('No code or traceback cells found.', 'craftContext');
  enterStaging(ids, 'Code + Errors');
};

const lensLastN = (n = null) => {
  let count = n;
  if (count == null) {
    const raw = prompt('Include last N cells in context:', '5');
    if (raw == null) return;
    count = Math.max(1, parseInt(raw, 10) || 5);
  }
  const cells = allCells().slice(-count);
  if (!cells.length) return;
  enterStaging(cells.map(c => c.id), `Last ${cells.length}`);
};

## Context Budget Meter, Preview Modal & Traceback Quick-Fix (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftContext
#| eval: false
/* ── Context Budget Meter (1.4) & Heat Strip ───────────────────────── */
let meter = null;
let heatEnabled = store.get(HEAT_KEY, '0') === '1';

function mountMeter() {
  if (!meter) {
    meter = mountTpl('ctx_meter', {refs: true});
    on(meter.root, 'click', e => { e.preventDefault(); e.stopPropagation(); openPreview(); }, sig);
  }
  const nav = qs('.uk-container > nav.monster-navbar');
  if (nav && !nav.contains(meter.root)) {
    const right = nav.querySelector('.uk-navbar-right, :scope > div:last-child') || nav;
    right.prepend(meter.root);
  } else if (!nav && !meter.root.isConnected && document.body) {
    document.body.append(meter.root);
  }
}

function updateMeter() {
  mountMeter();
  const cells = allCells();
  let activeTok = 0, hiddenTok = 0;
  const cellList = [];
  for (const c of cells) {
    const t = cellTokens(c);
    const hid = isHiddenFromAI(c);
    if (hid) hiddenTok += t;
    else activeTok += t;
    cellList.push({c, t, hid});
  }
  if (meter?.val) {
    const nextTxt = `~${fmtTok(activeTok)} tok`;
    if (meter.val.textContent !== nextTxt) meter.val.textContent = nextTxt;
    const lvl = activeTok >= 120000 ? 'high' : activeTok >= 50000 ? 'warn' : 'ok';
    if (meter.root.dataset.level !== lvl) meter.root.dataset.level = lvl;
    const nextTitle = `Active AI context: ~${activeTok.toLocaleString()} tokens (${hiddenTok.toLocaleString()} hidden) · Click to preview`;
    if (meter.root.title !== nextTitle) meter.root.title = nextTitle;
  }
  if (document.documentElement.classList.contains('craft-ctx-heat-on') !== heatEnabled) {
    document.documentElement.classList.toggle('craft-ctx-heat-on', heatEnabled);
  }
  for (const {c, t, hid} of cellList) {
    if (heatEnabled && !hid && activeTok > 0) {
      const share = Math.min(1, t / Math.max(activeTok, 1));
      const heatVal = (0.12 + share * 0.88).toFixed(2);
      if (c.style.getPropertyValue('--craft-heat') !== heatVal) c.style.setProperty('--craft-heat', heatVal);
      if (!c.classList.contains('craft-ctx-heat-cell')) c.classList.add('craft-ctx-heat-cell');
    } else {
      if (c.classList.contains('craft-ctx-heat-cell')) c.classList.remove('craft-ctx-heat-cell');
      if (c.style.getPropertyValue('--craft-heat')) c.style.removeProperty('--craft-heat');
    }
  }
}

/* ── "What the AI Sees" Preview Modal (1.5) ────────────────────────── */
let prevModal = null;

function renderPreview() {
  if (!prevModal) return;
  const cells = allCells();
  let activeTok = 0, includedCount = 0;
  const rows = cells.map((c, idx) => {
    const hid = isHiddenFromAI(c);
    const pin = isPinned(c);
    const tok = cellTokens(c);
    if (!hid) { activeTok += tok; includedCount++; }
    const mtype = cellMtype(c);
    const snippet = (c.querySelector(`${SOLVEIT.sel.cardIn} ${SOLVEIT.sel.msgContent}`)?.textContent || '').replace(/\s+/g, ' ').trim().slice(0, 90) || '(empty)';
    return el('div', {cls: `craft-ctx-prev-row${hid ? ' is-hidden' : ''}${pin ? ' is-pinned' : ''}`},
      el('span', {cls: 'craft-ctx-prev-idx'}, `#${idx + 1}`),
      el('span', {cls: `craft-ctx-prev-type type-${mtype}`}, mtype),
      pin ? el('span', {cls: 'craft-ctx-prev-pin', title: 'Pinned'}, '📌') : null,
      el('span', {cls: 'craft-ctx-prev-snip'}, snippet),
      el('span', {cls: 'craft-ctx-prev-tok'}, `~${fmtTok(tok)}`),
      el('button', {
        type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs craft-ctx-prev-tog',
        onClick: async () => { await setHiddenFromAI(c, !hid); renderPreview(); updateMeter(); },
      }, hid ? 'Include' : 'Hide'),
    );
  });
  prevModal.total.textContent = `${includedCount}/${cells.length} cells · ~${activeTok.toLocaleString()} tokens`;
  prevModal.list.replaceChildren(...(rows.length ? rows : [el('div', {cls: 'craft-empty-state'}, 'No cells in dialog.')]));
}

function openPreview() {
  if (!prevModal) {
    prevModal = mountModal('ctx_preview_modal', {refs: true, signal: ac.signal});
    on(prevModal.done, 'click', () => prevModal.close(), sig);
    on(prevModal.heat, 'click', () => {
      heatEnabled = !heatEnabled;
      store.set(HEAT_KEY, heatEnabled ? '1' : '0');
      updateMeter();
    }, sig);
    on(prevModal.savePreset, 'click', () => {
      const ids = allCells().filter(c => !isHiddenFromAI(c)).map(c => c.id);
      const name = prompt('Save currently included cells as preset:');
      if (name && savePreset(name, ids)) notify(`Saved preset "${name}".`, 'success');
    }, sig);
  }
  renderPreview();
  prevModal.open();
}

/* ── Traceback Quick-Fix (1.6) ─────────────────────────────────────── */
async function askAiAboutTraceback(codeCell) {
  if (!codeCell) return;
  const codeId = normId(codeCell.id);
  const before = new Set(allCells().map(c => c.id));
  try {
    await htmxPost(SOLVEIT.api.addBelow, {id_: codeCell.id, after: ''});
  } catch (err) {
    notifyErr('Could not insert prompt cell below traceback.', 'craftContext');
    return;
  }
  const newCell = await waitForNewCell(before, 3000);
  if (!newCell) return;
  const newId = normId(newCell.id);
  await CORE.switchCellMtype({id: newId, mtyp: 'prompt', tag: 'craftContext'});
  const promptText = 'Explain and fix the traceback error in the code cell above.';
  await htmxPost(SOLVEIT.api.setContents, {id_: '_' + newId, ids: newId, contents: JSON.stringify({[newId]: promptText})});
  enterStaging([codeId, newId], 'Traceback Fix');
  newCell.scrollIntoView({block: 'nearest'});
}

function syncTracebackButtons(cells = allCells()) {
  const tbSel = `:has(${SOLVEIT.sel.ansiRed}, .ansi-red-intense-fg)`;
  for (const cell of cells) {
    const out = cell.querySelector(SOLVEIT.sel.cardOut);
    if (!out || !out.matches(tbSel) || out.querySelector(':scope > .craft-tb-fix-btn')) continue;
    const btn = mountTpl('ctx_tb_fix_btn');
    on(btn, 'click', e => {
      e.preventDefault(); e.stopPropagation();
      askAiAboutTraceback(cell);
    }, sig);
    out.prepend(btn);
  }
}

## Menus, Hotkeys, Observer & Public API (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftContext
#| eval: false
/* ── Action Menu & Hotkey Registration ─────────────────────────────── */
const offMenu = window.craftActionMenus?.registerProvider?.(ctx => {
  if (ctx.isOutput || !ctx.cell) return [];
  return [
    {header: 'Context Control'},
    {label: 'All-But this cell (one-shot)', onClick: () => lensAllBut(ctx.cell)},
    {label: 'Lens: This heading section', onClick: () => lensHeadingSection(ctx.cell)},
    {label: 'Lens: Pinned + this cell', onClick: () => lensPinnedPlus(ctx.cell)},
    {label: 'Lens: Code + errors only', onClick: () => lensCodeAndErrors()},
    {label: 'What the AI sees…', onClick: () => openPreview()},
  ];
});

const reg = spec => window.__hotkeyRegister?.(spec);
reg({id: 'ctx-all-but', label: 'All-But this cell (one-shot minimal context)', group: 'CRAFT · Context', combo: '', handler: () => { lensAllBut(); return true; }});
reg({id: 'ctx-lens-section', label: 'Context Lens: This heading section only', group: 'CRAFT · Context', combo: '', handler: () => { lensHeadingSection(); return true; }});
reg({id: 'ctx-lens-pinned', label: 'Context Lens: Pinned cells + this cell', group: 'CRAFT · Context', combo: '', handler: () => { lensPinnedPlus(); return true; }});
reg({id: 'ctx-lens-code-err', label: 'Context Lens: Code cells + error outputs only', group: 'CRAFT · Context', combo: '', handler: () => { lensCodeAndErrors(); return true; }});
reg({id: 'ctx-lens-last-n', label: 'Context Lens: Last N cells', group: 'CRAFT · Context', combo: '', handler: () => { lensLastN(); return true; }});
reg({id: 'ctx-preview', label: 'What the AI sees (context preview)', group: 'CRAFT · Context', combo: '', handler: () => { openPreview(); return true; }});
reg({id: 'ctx-heat-toggle', label: 'Toggle context token heat strip', group: 'CRAFT · Context', combo: '', handler: () => { heatEnabled = !heatEnabled; store.set(HEAT_KEY, heatEnabled ? '1' : '0'); updateMeter(); return true; }});
reg({id: 'ctx-preset-save', label: 'Save context preset (visible cells)', group: 'CRAFT · Context', combo: '', handler: () => {
  const ids = (staging ? [...staging.scopeIds] : allCells().filter(c => !isHiddenFromAI(c)).map(c => c.id));
  const name = prompt('Preset name:');
  if (name && savePreset(name, ids)) notify(`Saved preset "${name}".`, 'success');
  return true;
}});
reg({id: 'ctx-preset-load', label: 'Load context preset (one-shot All-But)', group: 'CRAFT · Context', combo: '', handler: () => {
  const presets = loadPresets(), names = Object.keys(presets);
  if (!names.length) return (notifyErr('No saved presets for this dialog.', 'craftContext'), true);
  const pick = prompt(`Enter preset name to stage (${names.join(', ')}):`, names[0]);
  if (pick && presets[pick]) enterStaging(presets[pick], `Preset: ${pick}`);
  return true;
}});
reg({id: 'ctx-preset-apply', label: 'Apply context preset permanently', group: 'CRAFT · Context', combo: '', handler: () => {
  const presets = loadPresets(), names = Object.keys(presets);
  if (!names.length) return (notifyErr('No saved presets for this dialog.', 'craftContext'), true);
  const pick = prompt(`Enter preset name to apply permanently (${names.join(', ')}):`, names[0]);
  if (pick && presets[pick]) applyPermanent(presets[pick]);
  return true;
}});
reg({id: 'ctx-preset-del', label: 'Delete context preset', group: 'CRAFT · Context', combo: '', handler: () => {
  const presets = loadPresets(), names = Object.keys(presets);
  if (!names.length) return (notifyErr('No saved presets for this dialog.', 'craftContext'), true);
  const pick = prompt(`Delete preset (${names.join(', ')}):`, names[0]);
  if (pick && deletePreset(pick)) notify(`Deleted preset "${pick}".`, 'success');
  return true;
}});

const offCells = onCells(cells => {
  syncTracebackButtons(cells);
  if (staging) syncStagingUI();
}, {attrs: ['class']});
const offMeter = onDomChange(updateMeter, {inDialog: true, attrs: ['class'], delay: 200});

updateMeter();
syncTracebackButtons();
checkRecoveryBanner();

window.craftContext = {
  allCells, isHiddenFromAI, isPinned, cellTokens, setHiddenFromAI, getSectionCells,
  stageScope: enterStaging, cancelStaging, applyPermanent, restoreFromSnapshot,
  lensAllBut, lensHeadingSection, lensPinnedPlus, lensCodeAndErrors, lensLastN,
  loadPresets, savePreset, deletePreset, openPreview, updateMeter, askAiAboutTraceback,
  disable() {
    ac.abort();
    offCells?.();
    offMeter?.();
    offMenu?.();
    restoreWatch?.();
    clearStagingVisuals();
    stagingBar.destroy();
    recoverBar.destroy();
    meter?.root?.remove();
    prevModal?.destroy?.();
    qsa('.craft-tb-fix-btn').forEach(b => b.remove());
    qsa('.craft-ctx-heat-cell').forEach(c => { c.classList.remove('craft-ctx-heat-cell'); c.style.removeProperty('--craft-heat'); });
    delete window.craftContext;
  },
};

## Context Control CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Context Control, All-But, Budget Meter & Preview (13_context_control) ── */
html.craft-ab-staging #dialog-container > .editable:not(.craft-ab-included) {
  opacity: 0.28 !important;
  filter: grayscale(0.45);
  transition: opacity .15s ease;
  cursor: pointer;
}
html.craft-ab-staging #dialog-container > .editable.craft-ab-included {
  box-shadow: inset 3px 0 0 var(--craft-accent) !important;
  cursor: pointer;
}
#craft-ab-bar, #craft-ab-recover {
  position: fixed;
  left: 50%;
  bottom: calc(16px + env(safe-area-inset-bottom, 0px));
  transform: translateX(-50%);
  z-index: 1220;
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  gap: 8px;
  max-width: min(94vw, 820px);
  padding: 7px 12px;
  border-radius: 999px;
  background: var(--craft-recessed);
  border: 1px solid var(--craft-accent-border);
  box-shadow: 0 12px 32px rgba(0,0,0,.55);
  color: var(--craft-fg);
  font-size: 11.5px;
}
#craft-ab-bar[hidden], #craft-ab-recover[hidden] { display: none; }
html.craft-mobile #craft-ab-bar, html.craft-mobile #craft-ab-recover {
  bottom: calc(64px + env(safe-area-inset-bottom, 0px));
  border-radius: 12px;
}
#craft-ab-recover { border-color: var(--craft-amber); }
.craft-ab-summary { font-weight: 500; color: var(--craft-fg-strong); }
.craft-ab-preset { width: auto !important; height: 24px !important; padding: 0 22px 0 8px !important; font-size: 11px !important; }

/* Context Budget Meter */
.craft-ctx-meter {
  display: inline-flex !important;
  align-items: center;
  justify-content: center;
  flex: 0 0 auto;
  gap: 4px;
  width: auto !important;
  min-width: max-content;
  height: 22px !important;
  line-height: 1 !important;
  padding: 0 9px !important;
  margin: 0 4px 0 0 !important;
  box-sizing: border-box;
  white-space: nowrap !important;
  border-radius: 999px;
  border: 1px solid var(--craft-border);
  background: var(--craft-surface-alt);
  color: var(--craft-muted);
  font-family: var(--craft-code-family);
  font-size: 11px !important;
  cursor: pointer;
  transition: border-color .12s ease, color .12s ease;
}
.craft-ctx-meter-val { white-space: nowrap !important; line-height: 1 !important; }
.craft-ctx-meter:hover { border-color: var(--craft-accent); color: var(--craft-fg-strong); }
.craft-ctx-meter[data-level="warn"] { color: var(--craft-amber); border-color: var(--craft-amber); }
.craft-ctx-meter[data-level="high"] { color: var(--craft-red); border-color: var(--craft-red); }

/* Heat strip on active cells */
html.craft-ctx-heat-on #dialog-container > .editable.craft-ctx-heat-cell > .card-in {
  border-left: 3px solid color-mix(in srgb, var(--craft-amber) calc(var(--craft-heat, 0.2) * 100%), transparent) !important;
}

/* "What the AI Sees" Preview Modal */
.craft-ctx-prev-card {
  width: min(720px, 92vw);
  max-height: 76vh;
  display: flex;
  flex-direction: column;
  overflow: hidden;
}
.craft-modal-head {
  display: flex;
  align-items: center;
  gap: 10px;
  padding: 12px 16px;
  border-bottom: 1px solid var(--craft-border);
  font-size: 13px;
}
.craft-modal-x {
  margin-left: auto;
  border: 0;
  background: transparent;
  color: var(--craft-muted);
  font-size: 18px;
  line-height: 1;
  cursor: pointer;
  padding: 2px 6px;
  border-radius: 4px;
}
.craft-modal-x:hover { color: var(--craft-fg-strong); background: var(--craft-hover); }
.craft-ctx-prev-total { font-family: var(--craft-code-family); font-size: 11px; color: var(--craft-accent); }
.craft-ctx-prev-note {
  padding: 8px 16px;
  font-size: 11px;
  color: var(--craft-meta-fg, var(--craft-muted));
  background: var(--craft-recessed);
  border-bottom: 1px solid var(--craft-border-soft);
}
.craft-ctx-prev-list { overflow-y: auto; padding: 8px 12px; display: flex; flex-direction: column; gap: 4px; }
.craft-ctx-prev-row {
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 5px 8px;
  border-radius: 5px;
  background: var(--craft-bg);
  border: 1px solid var(--craft-border-soft);
  font-size: 11.5px;
}
.craft-ctx-prev-row.is-hidden { opacity: 0.45; }
.craft-ctx-prev-idx { font-family: var(--craft-code-family); font-size: 10px; color: var(--craft-muted); width: 26px; flex: none; }
.craft-ctx-prev-type {
  font-size: 10px; text-transform: uppercase; padding: 1px 5px; border-radius: 3px;
  background: var(--craft-hover); color: var(--craft-fg); flex: none;
}
.craft-ctx-prev-type.type-code { color: var(--craft-cyan); }
.craft-ctx-prev-type.type-prompt { color: var(--craft-purple); }
.craft-ctx-prev-type.type-note { color: var(--craft-green); }
.craft-ctx-prev-snip { flex: 1; min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.craft-ctx-prev-tok { font-family: var(--craft-code-family); font-size: 10.5px; color: var(--craft-muted); flex: none; }
.craft-modal-foot {
  display: flex; align-items: center; justify-content: flex-end; gap: 8px;
  padding: 10px 16px; border-top: 1px solid var(--craft-border);
}

/* Traceback Quick-Fix button */
.craft-tb-fix-btn {
  display: inline-flex;
  align-items: center;
  gap: 4px;
  margin: 4px 0 2px 14px;
  padding: 2px 8px;
  border-radius: 4px;
  border: 1px solid var(--craft-red);
  background: color-mix(in srgb, var(--craft-red) 14%, var(--craft-recessed));
  color: var(--craft-fg-strong);
  font-size: 10.5px;
  cursor: pointer;
}
.craft-tb-fix-btn:hover { background: color-mix(in srgb, var(--craft-red) 28%, var(--craft-recessed)); }

Verify `13_context_control.ipynb` exports `window.craftContext`, its templates, and CSS.

In [ ]:
import os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/13_context_control.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftContext', 'ctx_staging_bar', 'ctx_preview_modal', 'ctx_tb_fix_btn', 'lensHeadingSection', 'askAiAboutTraceback'):
    assert sym in js, f"Missing {sym}"
assert '#craft-ab-bar' in css and '.craft-ctx-meter' in css and '.craft-tb-fix-btn' in css

node_script = r'''
const assert = require('assert');
const src = process.env.CTX_JS;
const i0 = src.indexOf('const parseTokenNum =');
const i1 = src.indexOf('const cellTokens =');
const j0 = src.indexOf('const fmtTok =');
const j1 = src.indexOf('async function setHiddenFromAI');
const {parseTokenNum, fmtTok} = new Function(
  src.slice(i0, i1) + '\n' + src.slice(j0, j1) + '\nreturn {parseTokenNum, fmtTok};'
)();
assert.strictEqual(parseTokenNum('42'), 42);
assert.strictEqual(parseTokenNum('1,234'), 1234);
assert.strictEqual(parseTokenNum('1.5k'), 1500);
assert.strictEqual(parseTokenNum('~12.3K tok'), 12300);
assert.strictEqual(parseTokenNum('2.5m'), 2500000);
assert.strictEqual(parseTokenNum(''), 0);
assert.strictEqual(parseTokenNum(null), 0);
assert.strictEqual(parseTokenNum('no digits'), 0);
assert.strictEqual(fmtTok(999), '999');
assert.strictEqual(fmtTok(1500), '1.5k');
'''
subprocess.run(['node', '-e', node_script], env={**os.environ, 'CTX_JS': js}, check=True)
